### Loading the dataset

We first load the dataset, where each entry contains an instrcution along with corresponding input and output

In [1]:
import json
import os
import urllib

In [2]:
def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode('utf-8')
        with open (file_path, "w", encoding='utf-8') as file:
            file.write(text_data)
            
    else:
        with open(file_path, 'r', encoding='utf-8') as file:
            text_data = file.read()
    with open(file_path, "r") as file:
        data = json.load(file)
    return data
            

In [3]:
file_path = "instruction-data.json"
url = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch""/main/ch07/01_main-chapter-code/instruction-data.json")
data = download_and_load_file(file_path, url)
print("Number of entries:", len(data))
print(data[1]);

Number of entries: 1100
{'instruction': 'Edit the following sentence for grammar.', 'input': 'He go to the park every day.', 'output': 'He goes to the park every day.'}


Let’s define a format_input function that we can use to convert the entries in the
data list into the Alpaca-style input format.

In [4]:
def format_input(entry):
    
    instruction_text = (
    f"Below is an instruction that describes a task. "
    f"Write a response that appropriately completes the request."
    f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = (f"\n\n### Input:\n{entry['input']}" if entry["input"] else "")
    return instruction_text + input_text

In [5]:
model_input = format_input(data[999])
desired_response= f"\n\n### Response:\n {data[50]['output']}"
print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?

### Response:
 The correct spelling is 'Occasion.'


In [6]:
train_portion= int(len(data)* 0.85) 
test_portion= int(len(data)* 0.1) 
val_portion= len(data) - train_portion - test_portion
train_data = data[:train_portion]
test_data= data[train_portion : train_portion + test_portion]
val_data= data[train_portion+ test_portion:]

print("Training set of length: ", len(train_data))
print("Validation set of length ", len(val_data))
print("Test set of length: ", len(test_data))



Training set of length:  935
Validation set of length  55
Test set of length:  110


the batching process for instruction fine-tuning is a bit more involved
and requires us to create our own custom collate function that we will later plug into

We will 
pad the data samples to equal lengths
so we can assemble multiple instruction
examples in a batch.
Then, we create the PyTorch
data loaders we will use for
the DataLoader. We implement this custom collate function to handle the specific
requirements and formatting of our instruction fine-tuning dataset.

In [7]:
import torch
from torch.utils.data import Dataset


class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts= []
        for entry in data:
            instruction_plus_input= format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(tokenizer.encode(full_text))
    def __getitem__(self, index):
        return self.encoded_texts[index]
    
    def __len__(self):
        return len(self.data)


We now create a custom collate function that we can pass through the dataloader. This custom function pads all example in some batch to be of equal length while different entries in different batches can have different lengths i.e only the entries in same batch should be padded to be of equal length

In [8]:
def custom_collate_1(batch, pad_token_id = 5056, device= "cpu"):
    batch_max_length= max(len(item) + 1 for item in batch)
    inputs_arr = []
    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        padded = (new_item + [pad_token_id] * (batch_max_length - len(new_item)))
        inputs = torch.tensor(padded[:-1])
        inputs_arr.append(inputs)
        
    inputs_tensor= torch.stack(inputs_arr).to(device)
    return inputs_tensor

In [9]:
inputs_1 = [0,1,2,3,4]
inputs_2= [5,6]
inputs_3= [7,8,9]
batch = (inputs_1, inputs_2, inputs_3)
print(custom_collate_1(batch))


tensor([[   0,    1,    2,    3,    4],
        [   5,    6, 5056, 5056, 5056],
        [   7,    8,    9, 5056, 5056]])


However we also need to create batches with the target token id corresponding to the batch of input ids. The target token ids match the input ids but are shifted by one position

In [10]:
def custom_collate_2(batch, pad_token_id=50256, device="cpu"):
    batch_max_length= max(len(item) + 1 for item in batch)
    inputs_arr = []
    target_arr = []
    for item in batch:
        new_item = item.copy();
        new_item += [pad_token_id]
        padded = (new_item + [pad_token_id] * (batch_max_length - len(new_item)))
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])
        inputs_arr.append(inputs)
        target_arr.append(targets)
    inputs_tensor = torch.stack(inputs_arr).to(device)
    targets_tensor = torch.stack(target_arr).to(device)
    return inputs_tensor, targets_tensor

inputs,targets = custom_collate_2(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256, 50256, 50256, 50256],
        [    8,     9, 50256, 50256, 50256]])


In the next step, we assign a -100 placeholder value to all padding token. This special value allows us to exclude these padding tokens
from contributing to the training loss calculation, ensuring that only meaningful data
influences model learning. However, note that we retain one end-of-text token, ID 50256, in the target list. Retaining it allows the LLM to learn when to generate an end-
of-text token in response to instructions, which we use as an indicator that the gener-
ated response is complete.

In [11]:
def custom_collate(batch, pad_token_id = 50256, ignore_index=-100, allowed_max_length=None, device="cpu"):
    batch_max_length = max(len(item) + 1 for item in batch)
    inputs_arr, targets_arr = [], []
    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id];
        padded = (new_item + [pad_token_id] * (batch_max_length - len(new_item)))
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index #basically replace all but the first occurence of the pad token with -100
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]  
            targets = targets[:allowed_max_length]
        inputs_arr.append(inputs)
        targets_arr.append(targets)
    inputs_tensor = torch.stack(inputs_arr).to(device)
    targets_tensor = torch.stack(targets_arr).to(device)
    return inputs_tensor, targets_tensor      

In [12]:
inputs, targets = custom_collate(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]])


To understand why we use -100 as the ignore index consider the following example

In [13]:
logits_1 = torch.tensor([[-1.0, 1.0], [-0.5, 1.5]])
targets_1 = torch.tensor([0,1])#correct token index
loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)
print(loss_1)
logits_2 = torch.tensor(
[[-1.0, 1.0],[-0.5, 1.5],[-0.5, 1.5]])
targets_2 = torch.tensor([0, 1, 1])
loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
print(loss_2)
#We cann see that adding values definitely has an impact on the cross entropy loss
targets_3 = torch.tensor([0, 1, -100])
loss_3 = torch.nn.functional.cross_entropy(logits_2, targets_3)
print(loss_3)
print("loss_1 == loss_3:", loss_1 == loss_3)

#The  cross entropy loss ignored the 3rd entry which is -100 becuase by default in pytorch the cross entropy ignores labels with -100 values 

tensor(1.1269)
tensor(0.7936)
tensor(1.1269)
loss_1 == loss_3: tensor(True)


In addition to masking out padding tokens, it is also common to mask out the tar-
get token IDs that correspond to the instructions. By mask-
ing out the LLM’s target token IDs corresponding to the instruction, the cross
entropy loss is only computed for the generated response target IDs. Thus, the model
is trained to focus on generating accurate responses rather than memorizing instruc-
tions, which can help reduce overfitting

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


Next, to reuse the chosen device setting in custom_collate_fn when we plug it
into the PyTorch DataLoader class, we use the partial function from Python’s
functools standard library to create a new version of the function with the device
argument prefilled. Additionally, we set the allowed_max_length to 1024, which trun-
cates the data to the maximum context length supported by the GPT-2 model

In [15]:
from functools import partial
customized_collat_fn =partial(custom_collate, device= device, allowed_max_length=1024)

Next we setup the dataloaders

In [16]:
from torch.utils.data import DataLoader
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
num_workers=0
batch_size= 8
torch.manual_seed(123)
train_dataset = InstructionDataset(train_data, tokenizer)
train_loader= DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn = customized_collat_fn,
    shuffle=True,
    drop_last = True,
    num_workers=num_workers
)
val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
val_dataset,
batch_size=batch_size,
collate_fn=customized_collat_fn,
shuffle=False,
drop_last=False,
num_workers=num_workers
)
test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
test_dataset,
batch_size=batch_size,
collate_fn=customized_collat_fn,
shuffle=False,
drop_last=False,
num_workers=num_workers
)
print("Train loader:")
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)


Train loader:
torch.Size([8, 61]) torch.Size([8, 61])
torch.Size([8, 76]) torch.Size([8, 76])
torch.Size([8, 73]) torch.Size([8, 73])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 72]) torch.Size([8, 72])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 62]) torch.Size([8, 62])
torch.Size([8, 75]) torch.Size([8, 75])
torch.Size([8, 62]) torch.Size([8, 62])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 77]) torch.Size([8, 77])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 79]) torch.Size([8, 79])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 66]) torch.Size([8, 66])
torch.Size([8, 83]) torch.Size([8, 83])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 68]) torch.

### Loading pretrained model

we load the medium-sized model with 355 million parameter

In [17]:
from gpt_download import download_and_load_gpt2
from gpt_model import GPTModel
from load_weights_into_gpt import load_weights_into_gpt
BASE_CONFIG = {
"vocab_size": 50257, 
"context_length": 1024, 
"drop_rate": 0.0, 
"qkv_bias": True 
}
model_configs = {
"gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
"gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
"gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
"gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}
CHOOSE_MODEL = "gpt2-medium (355M)"
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])
model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(
model_size=model_size,
models_dir="gpt2"
)
model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval();

checkpoint: 100%|██████████| 77.0/77.0 [00:00<00:00, 67.0kiB/s]
encoder.json: 100%|██████████| 1.04M/1.04M [00:01<00:00, 528kiB/s] 
hparams.json: 100%|██████████| 91.0/91.0 [00:00<00:00, 159kiB/s]
model.ckpt.data-00000-of-00001: 100%|██████████| 1.42G/1.42G [26:53<00:00, 879kiB/s]   
model.ckpt.index: 100%|██████████| 10.4k/10.4k [00:00<00:00, 10.6MiB/s]
model.ckpt.meta: 100%|██████████| 927k/927k [00:02<00:00, 447kiB/s]  
vocab.bpe: 100%|██████████| 456k/456k [00:01<00:00, 247kiB/s]  
2026-02-21 15:11:15.020609: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 205852672 exceeds 10% of free system memory.


In [18]:
torch.manual_seed(123)
input_text = format_input(val_data[0])
print(input_text)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'


In [19]:
from generate_text import generate, text_to_token, token_to_text
token_ids = generate(model=model,idx=text_to_token(input_text, tokenizer),max_new_tokens=35,context_size=BASE_CONFIG["context_length"],eos_id=50256,
)
generated_text = token_to_text(token_ids, tokenizer)

The generate function returns the combined input and output text. This behavior was
previously convenient since pretrained LLMs are primarily designed as text-completion
models, where the input and output are concatenated to create coherent and legible
text. However, when evaluating the model’s performance on a specific task, we often
want to focus solely on the model’s generated response.
To isolate the model’s response text, we need to subtract the length of the input
instruction from the start of the generated_text:

In [ ]:
response_text= generated_text[len(input_text):].strip()
print(response_text)


Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'

### Response:

The chef cooks the meal every day.

### Instruction:

Convert the active sentence to passive: 'The chef cooks the


### Finetuning the llm for instructions

In [22]:
from model_training_utlilities import calc_loss_loader, train_model_simple


In [23]:
model.to(device)
torch.manual_seed(123)
with torch.no_grad():
    train_loss= calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
    
print("Training loss: ", train_loss)
print("Val loss: ", val_loss)

Training loss:  tensor(3.8249)
Val loss:  tensor(3.7610)


In [ ]:
import time 
start_time= time.time()
torch.manual_seed(123)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.00005, weight_decay=0.1)
num_epochs= 2
train_losses, val_losses, tokens_seen = train_model_simple(
model, train_loader, val_loader, optimizer, device,
num_epochs=num_epochs, eval_freq=5, eval_iter=5,
start_context=format_input(val_data[0]), tokenizer=tokenizer
)
end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")